# AI-Based Smart Electricity Demand Forecasting and Dynamic Energy Allocation

> **Hackathon End-to-End Prototype & Project Notebook**  
> Combining Machine Learning Short-Term Demand Forecasting with Priority-Based Dynamic Energy Allocation, Demand Surge/Generation Drop Simulations, and Interactive Dashboard Telemetry.

---
## Table of Contents
1. **Project Overview & System Architecture**
2. **Step 1: Environment Setup & Library Imports**
3. **Step 2: Synthetic Energy Dataset Generation (`data_generator.py`)**
4. **Step 3: Exploratory Data Analysis & Feature Engineering**
5. **Step 4: AI/ML Demand Forecasting (`demand_forecasting.py`)**
6. **Step 5: Load Classification & Dynamic Energy Allocation (`energy_allocation.py`)**
7. **Step 6: Hackathon Scenario Simulations (Surge & Generation Drop)**
8. **Step 7: Flask Backend API Implementation (`app.py`)**
9. **Step 8: Full Frontend Dashboard Code (HTML, CSS, JS)**
10. **Step 9: Testing, Verification & Deployment**

---
## 1. Environment Setup & Library Imports

Install dependencies if running in Colab or a fresh environment:
```bash
!pip install Flask numpy pandas scikit-learn matplotlib seaborn
```

In [ ]:
import os
import sys
import copy
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Configure visual plot styling
plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
print('Libraries imported successfully!')

---
## 2. Synthetic Energy Dataset Generation (`data_generator.py`)

Generates realistic electricity demand data reflecting diurnal patterns, temperature-driven HVAC spikes, occupancy fluctuations, and user behavior.
*Note: Synthetic data calibrated for hackathon demonstration.*

In [ ]:
def generate_energy_dataset(n_samples: int = 2000, output_path: str = 'data/energy_data.csv') -> pd.DataFrame:
    np.random.seed(42)

    # Time & behavioral variables
    hours = np.random.randint(0, 24, size=n_samples)
    day_types = np.random.choice([0, 1], size=n_samples, p=[0.71, 0.29])  # 0: Weekday, 1: Weekend
    user_activity = np.random.choice([0, 1, 2], size=n_samples, p=[0.25, 0.50, 0.25])  # 0: Low, 1: Med, 2: High

    # Environmental variables (warmer afternoons)
    temp_base = 24.0 + 8.0 * np.sin((hours - 8) / 24.0 * np.pi)
    temperatures = np.clip(temp_base + np.random.normal(0, 4, size=n_samples), 16.0, 42.0)
    humidity = np.clip(60.0 - (temperatures - 25.0) * 0.8 + np.random.normal(0, 8, size=n_samples), 25.0, 95.0)

    # Occupancy depends on diurnal cycle and weekday/weekend
    occupancy_base = np.where(
        day_types == 0,
        np.where((hours >= 8) & (hours <= 18), 75.0, 25.0),
        np.where((hours >= 10) & (hours <= 20), 60.0, 40.0)
    )
    occupancy = np.clip(occupancy_base + np.random.normal(0, 12, size=n_samples), 5.0, 100.0)

    # Previous demand (baseline with diurnal cycle)
    prev_demand_base = 50.0 + 25.0 * np.sin((hours - 6) / 24.0 * 2 * np.pi)
    previous_demand = np.clip(prev_demand_base + np.random.normal(0, 8, size=n_samples), 35.0, 130.0)

    # Physics- & behavior-based demand calculation
    demand = 25.0 + 0.30 * previous_demand

    # Hour peak factor (morning ramp 9-12, evening peak 17-21)
    hour_factor = np.where((hours >= 9) & (hours <= 12), 15.0, 0.0) + \
                  np.where((hours >= 17) & (hours <= 21), 20.0, 0.0) + \
                  np.where((hours >= 1) & (hours <= 5), -10.0, 0.0)
    demand += hour_factor

    # Temperature cooling degree effect above 24°C
    cooling_effect = np.maximum(0.0, temperatures - 24.0) * 1.8
    demand += cooling_effect

    # Occupancy effect
    demand += (occupancy / 100.0) * 22.0

    # User activity tier effect
    demand += user_activity * 7.5

    # Weekday base commercial activity
    demand += (1 - day_types) * 6.0

    # Gaussian noise
    demand += np.random.normal(0, 2.5, size=n_samples)
    demand = np.clip(demand, 30.0, 160.0).round(2)

    df = pd.DataFrame({
        'previous_demand': previous_demand.round(2),
        'temperature': temperatures.round(1),
        'humidity': humidity.round(1),
        'occupancy': occupancy.round(1),
        'hour': hours,
        'day_type': day_types,
        'user_activity': user_activity,
        'demand': demand
    })

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f'Generated {len(df)} samples saved to {output_path}')
    return df

df_energy = generate_energy_dataset(2000)
df_energy.head()

---
## 3. Exploratory Data Analysis & Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Temperature vs Electricity Demand
axes[0, 0].scatter(df_energy['temperature'], df_energy['demand'], alpha=0.3, color='#38bdf8')
axes[0, 0].set_title('Temperature (°C) vs. Demand (kW)')
axes[0, 0].set_xlabel('Temperature (°C)')
axes[0, 0].set_ylabel('Electricity Demand (kW)')

# Plot 2: Average Hourly Demand Profile
hourly_avg = df_energy.groupby('hour')['demand'].mean()
axes[0, 1].plot(hourly_avg.index, hourly_avg.values, marker='o', color='#10b981', linewidth=2.5)
axes[0, 1].set_title('Diurnal Electricity Demand Curve (Hour of Day)')
axes[0, 1].set_xlabel('Hour (0 - 23)')
axes[0, 1].set_ylabel('Average Demand (kW)')
axes[0, 1].set_xticks(range(0, 24, 2))

# Plot 3: Occupancy vs Demand
axes[1, 0].scatter(df_energy['occupancy'], df_energy['demand'], alpha=0.3, color='#a855f7')
axes[1, 0].set_title('Occupancy (%) vs. Demand (kW)')
axes[1, 0].set_xlabel('Occupancy (%)')
axes[1, 0].set_ylabel('Electricity Demand (kW)')

# Plot 4: Demand Distribution Histogram
axes[1, 1].hist(df_energy['demand'], bins=25, color='#f59e0b', edgecolor='black', alpha=0.75)
axes[1, 1].set_title('Electricity Demand Distribution')
axes[1, 1].set_xlabel('Demand (kW)')
axes[1, 1].set_ylabel('Count')

plt.tight_layout()
plt.show()

---
## 4. AI/ML Demand Forecasting (`demand_forecasting.py`)

We train a `RandomForestRegressor` model to predict short-term demand based on lag demand, weather, occupancy, and behavioral patterns.

In [ ]:
FEATURE_COLUMNS = [
    'previous_demand',
    'temperature',
    'humidity',
    'occupancy',
    'hour',
    'day_type',
    'user_activity'
]

X = df_energy[FEATURE_COLUMNS]
y = df_energy['demand']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_split=4,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('=== MODEL PERFORMANCE METRICS ===')
print(f'Mean Absolute Error (MAE) : {mae:.2f} kW')
print(f'Root Mean Squared Error   : {rmse:.2f} kW')
print(f'R² Accuracy Score         : {r2:.4f} ({r2*100:.1f}%)')

# Save trained model
os.makedirs('models', exist_ok=True)
with open('models/demand_model.pkl', 'wb') as f:
    pickle.dump({'model': rf_model, 'metrics': {'mae': mae, 'r2_score': r2}}, f)
print('Model saved to models/demand_model.pkl')

In [ ]:
plt.figure(figsize=(10, 4))
importances = pd.Series(rf_model.feature_importances_, index=FEATURE_COLUMNS).sort_values(ascending=True)
importances.plot(kind='barh', color='#38bdf8', edgecolor='black')
plt.title('Random Forest Feature Importance in Electricity Demand Forecasting')
plt.xlabel('Relative Importance Score')
plt.show()

---
## 5. Load Classification & Dynamic Energy Allocation (`energy_allocation.py`)

### Load Categories:
1. **Critical Loads (Priority 1)**: Medical equipment (20 kW), Emergency systems (10 kW), Essential servers (15 kW). Must never be dropped.
2. **Important Loads (Priority 2)**: Lighting (15 kW), Workstations (15 kW), Essential HVAC (10 kW). Supplied immediately after critical.
3. **Flexible Loads (Priority 3)**: EV Fast Chargers (25 kW), Water Heating (15 kW), Auxiliary HVAC (15 kW), Washing (10 kW). Dynamically throttled/shifted during deficits to eliminate blackout risk.

In [ ]:
DEFAULT_LOADS = [
    {'id': 'load_1', 'name': 'Medical Equipment', 'category': 'Critical', 'priority': 1, 'is_flexible': False, 'required_kw': 20.0, 'allocated_kw': 20.0, 'status': 'Active'},
    {'id': 'load_2', 'name': 'Emergency Systems', 'category': 'Critical', 'priority': 1, 'is_flexible': False, 'required_kw': 10.0, 'allocated_kw': 10.0, 'status': 'Active'},
    {'id': 'load_3', 'name': 'Essential Servers', 'category': 'Critical', 'priority': 1, 'is_flexible': False, 'required_kw': 15.0, 'allocated_kw': 15.0, 'status': 'Active'},
    {'id': 'load_4', 'name': 'Lighting', 'category': 'Important', 'priority': 2, 'is_flexible': False, 'required_kw': 15.0, 'allocated_kw': 15.0, 'status': 'Active'},
    {'id': 'load_5', 'name': 'Computers & Workstations', 'category': 'Important', 'priority': 2, 'is_flexible': False, 'required_kw': 15.0, 'allocated_kw': 15.0, 'status': 'Active'},
    {'id': 'load_6', 'name': 'Essential HVAC', 'category': 'Important', 'priority': 2, 'is_flexible': False, 'required_kw': 10.0, 'allocated_kw': 10.0, 'status': 'Active'},
    {'id': 'load_7', 'name': 'EV Fast Chargers', 'category': 'Flexible', 'priority': 3, 'is_flexible': True, 'required_kw': 25.0, 'allocated_kw': 25.0, 'status': 'Active'},
    {'id': 'load_8', 'name': 'Water Heating System', 'category': 'Flexible', 'priority': 3, 'is_flexible': True, 'required_kw': 15.0, 'allocated_kw': 15.0, 'status': 'Active'},
    {'id': 'load_9', 'name': 'Non-Critical HVAC / Chillers', 'category': 'Flexible', 'priority': 3, 'is_flexible': True, 'required_kw': 15.0, 'allocated_kw': 15.0, 'status': 'Active'},
    {'id': 'load_10', 'name': 'Washing & Heavy Equipment', 'category': 'Flexible', 'priority': 3, 'is_flexible': True, 'required_kw': 10.0, 'allocated_kw': 10.0, 'status': 'Active'}
]

def scale_loads_to_demand(loads, target_demand):
    current_sum = sum(l['required_kw'] for l in loads)
    if current_sum <= 0 or target_demand <= 0:
        return loads
    scale_factor = target_demand / current_sum
    scaled = copy.deepcopy(loads)
    allocated_so_far = 0.0
    for i, l in enumerate(scaled):
        if i == len(scaled) - 1:
            l['required_kw'] = round(target_demand - allocated_so_far, 1)
        else:
            l['required_kw'] = round(l['required_kw'] * scale_factor, 1)
            allocated_so_far += l['required_kw']
        l['allocated_kw'] = l['required_kw']
    return scaled

def allocate_energy(available_generation: float, predicted_demand: float = None, loads: list = None, electricity_price_kwh: float = 0.16) -> dict:
    loads = copy.deepcopy(loads if loads else DEFAULT_LOADS)
    available_generation = float(max(0.0, available_generation))
    
    if predicted_demand is not None and predicted_demand > 0:
        loads = scale_loads_to_demand(loads, predicted_demand)

    total_required = round(sum(l['required_kw'] for l in loads), 1)
    critical_loads = [l for l in loads if l['category'] == 'Critical']
    important_loads = [l for l in loads if l['category'] == 'Important']
    flexible_loads = [l for l in loads if l['category'] == 'Flexible']

    req_critical = round(sum(l['required_kw'] for l in critical_loads), 1)
    req_important = round(sum(l['required_kw'] for l in important_loads), 1)
    req_flexible = round(sum(l['required_kw'] for l in flexible_loads), 1)

    shifted_loads, managed_loads = [], []
    remaining = available_generation

    # 1. Critical Loads Priority
    alloc_critical = 0.0
    for load in critical_loads:
        req = load['required_kw']
        if remaining >= req:
            load['allocated_kw'] = req
            load['status'] = 'Active'
            remaining -= req
        else:
            load['allocated_kw'] = round(max(0.0, remaining), 1)
            load['status'] = 'Protected Partial' if remaining > 0 else 'Offline'
            remaining = 0.0
        alloc_critical += load['allocated_kw']

    # 2. Important Loads Priority
    alloc_important = 0.0
    for load in important_loads:
        req = load['required_kw']
        if remaining >= req:
            load['allocated_kw'] = req
            load['status'] = 'Active'
            remaining -= req
        else:
            load['allocated_kw'] = round(max(0.0, remaining), 1)
            load['status'] = 'Reduced' if remaining > 0 else 'Offline'
            remaining = 0.0
        alloc_important += load['allocated_kw']

    # 3. Flexible Loads Dynamic Throttling/Shifting
    alloc_flexible = 0.0
    if len(flexible_loads) > 0:
        if remaining >= req_flexible:
            for load in flexible_loads:
                load['allocated_kw'] = load['required_kw']
                load['status'] = 'Active'
                remaining -= load['required_kw']
                alloc_flexible += load['allocated_kw']
        else:
            flex_budget = max(0.0, remaining)
            for load in flexible_loads:
                req = load['required_kw']
                name = load['name'].lower()
                if ('ev' in name or 'hvac' in name or 'chiller' in name) and flex_budget > 0:
                    portion = min(flex_budget, req * 0.50)
                    load['allocated_kw'] = round(portion, 1)
                    flex_budget -= load['allocated_kw']
                    load['status'] = 'Managed (Eco-Throttled)' if load['allocated_kw'] > 0 else 'Shifted (Off-Peak)'
                    (managed_loads if load['allocated_kw'] > 0 else shifted_loads).append(load['name'])
                elif flex_budget >= req * 0.40:
                    load['allocated_kw'] = round(req * 0.40, 1)
                    flex_budget -= load['allocated_kw']
                    load['status'] = 'Managed (Reduced)'
                    managed_loads.append(load['name'])
                else:
                    load['allocated_kw'] = 0.0
                    load['status'] = 'Shifted (Off-Peak)'
                    shifted_loads.append(load['name'])
                alloc_flexible += load['allocated_kw']

    all_loads = critical_loads + important_loads + flexible_loads
    total_allocated = round(alloc_critical + alloc_important + alloc_flexible, 1)
    shortage = round(max(0.0, total_required - available_generation), 1)
    surplus = round(max(0.0, available_generation - total_allocated), 1)

    # Cost Analysis
    off_peak_price = round(electricity_price_kwh * 0.55, 3)
    peak_penalty = round(electricity_price_kwh * 1.50, 3)
    unopt_cost = round(total_required * (peak_penalty if shortage > 0 else electricity_price_kwh), 2)
    opt_cost = round((total_allocated * electricity_price_kwh) + ((total_required - total_allocated) * off_peak_price), 2)
    saving = round(max(0.0, unopt_cost - opt_cost), 2)

    return {
        'critical': round(alloc_critical, 1),
        'important': round(alloc_important, 1),
        'flexible': round(alloc_flexible, 1),
        'total_required': total_required,
        'total_allocated': total_allocated,
        'available_generation': available_generation,
        'shortage': shortage,
        'surplus': surplus,
        'shifted_loads': shifted_loads,
        'managed_loads': managed_loads,
        'estimated_saving': saving,
        'loads': all_loads
    }

---
## 6. Hackathon Scenario Simulations

### Testing Demand Surge & Generation Drop Scenarios

In [ ]:
# Scenario 1: Normal Operation
res_normal = allocate_energy(available_generation=100.0, predicted_demand=94.0)
print('=== SCENARIO 1: NORMAL DEMAND ===')
print(f'Required: {res_normal["total_required"]} kW | Generation: {res_normal["available_generation"]} kW | Surplus: {res_normal["surplus"]} kW')

# Scenario 2: Sudden Demand Surge (+25 kW Spike)
res_surge = allocate_energy(available_generation=100.0, predicted_demand=125.0)
print('\n=== SCENARIO 2: DEMAND SURGE (125 kW) ===')
print(f'Required: {res_surge["total_required"]} kW | Generation: {res_surge["available_generation"]} kW | Shortage: {res_surge["shortage"]} kW')
print(f'Critical: {res_surge["critical"]} kW | Important: {res_surge["important"]} kW | Flexible: {res_surge["flexible"]} kW')
print(f'Managed Loads: {res_surge["managed_loads"]}')
print(f'Estimated Cost Savings: ${res_surge["estimated_saving"]}/hr')

# Scenario 3: Sudden Generation Drop (70 kW Supply Cut)
res_drop = allocate_energy(available_generation=70.0, predicted_demand=100.0)
print('\n=== SCENARIO 3: GENERATION DROP (70 kW) ===')
print(f'Required: {res_drop["total_required"]} kW | Generation: {res_drop["available_generation"]} kW | Shortage: {res_drop["shortage"]} kW')
print(f'Critical: {res_drop["critical"]} kW | Important: {res_drop["important"]} kW | Flexible: {res_drop["flexible"]} kW')
print(f'Shifted Loads: {res_drop["shifted_loads"]}')
print(f'Estimated Cost Savings: ${res_drop["estimated_saving"]}/hr')

In [ ]:
scenarios = ['Normal (94 kW)', 'Demand Surge (125 kW)', 'Generation Drop (70 kW)']
crit_vals = [res_normal['critical'], res_surge['critical'], res_drop['critical']]
imp_vals = [res_normal['important'], res_surge['important'], res_drop['important']]
flex_vals = [res_normal['flexible'], res_surge['flexible'], res_drop['flexible']]

x = np.arange(len(scenarios))
width = 0.55

plt.figure(figsize=(10, 5))
p1 = plt.bar(x, crit_vals, width, label='Critical Loads', color='#ef4444')
p2 = plt.bar(x, imp_vals, width, bottom=crit_vals, label='Important Loads', color='#f59e0b')
p3 = plt.bar(x, flex_vals, width, bottom=np.array(crit_vals) + np.array(imp_vals), label='Flexible Loads', color='#38bdf8')

plt.ylabel('Allocated Power (kW)')
plt.title('Dynamic Energy Allocation across Hackathon Scenarios')
plt.xticks(x, scenarios)
plt.legend()
plt.show()

---
## 7. Flask Web Application & REST APIs (`app.py`)

The complete Flask server exposing endpoints:
- `POST /predict`
- `POST /allocate`
- `POST /simulate/demand-surge`
- `POST /simulate/generation-drop`
- `GET /status`, `GET /loads`, `POST /reset`

In [ ]:
"""
AI Smart Energy Management System - Flask Backend & REST APIs.
Coordinates AI Demand Forecasting, Priority-Based Dynamic Energy Allocation,
Demand Surge and Generation Drop simulations, and real-time dashboard telemetry.
"""

import os
from flask import Flask, render_template, request, jsonify
from demand_forecasting import forecaster
from energy_allocation import allocate_energy, get_default_loads

BASE_DIR = os.path.abspath(os.path.dirname(__file__))
app = Flask(
    __name__,
    template_folder=os.path.join(BASE_DIR, "templates"),
    static_folder=os.path.join(BASE_DIR, "static")
)

# Preload or train ML model on startup
print("Initializing Demand Forecasting Model...")
forecaster.load_or_train()
print("Demand Forecasting Model ready.")

# Current in-memory operational state (can be reset)
system_state = {
    "current_demand": 80.0,
    "available_generation": 100.0,
    "temperature": 32.0,
    "humidity": 60.0,
    "occupancy": 75.0,
    "hour": 14,
    "day_type": "Weekday",
    "user_activity": "Medium",
    "last_prediction": 94.0,
    "last_allocation": None,
    "loads": get_default_loads(),
    "last_event": "System initialized normally."
}

# Run initial allocation so the dashboard loads with complete data
system_state["last_allocation"] = allocate_energy(
    available_generation=system_state["available_generation"],
    predicted_demand=system_state["last_prediction"],
    loads=system_state["loads"]
)


@app.route("/")
def index():
    """Serves the main interactive dashboard."""
    return render_template("index.html")


@app.route("/predict", methods=["POST"])
def predict():
    """
    POST /predict
    Receives current operating parameters, invokes the AI/ML model,
    and returns predicted demand, comparison with generation, and historical trend data.
    """
    try:
        data = request.get_json(force=True) or {}
        
        # Update system state with incoming parameters
        system_state["current_demand"] = float(data.get("current_demand", system_state["current_demand"]))
        system_state["available_generation"] = float(data.get("available_generation", system_state["available_generation"]))
        system_state["temperature"] = float(data.get("temperature", system_state["temperature"]))
        system_state["humidity"] = float(data.get("humidity", system_state["humidity"]))
        system_state["occupancy"] = float(data.get("occupancy", system_state["occupancy"]))
        system_state["hour"] = int(data.get("hour", system_state["hour"]))
        system_state["day_type"] = str(data.get("day_type", system_state["day_type"]))
        system_state["user_activity"] = str(data.get("user_activity", system_state["user_activity"]))

        # Execute ML prediction
        prediction_result = forecaster.predict({
            "current_demand": system_state["current_demand"],
            "temperature": system_state["temperature"],
            "humidity": system_state["humidity"],
            "occupancy": system_state["occupancy"],
            "hour": system_state["hour"],
            "day_type": system_state["day_type"],
            "user_activity": system_state["user_activity"]
        })

        pred_val = prediction_result["predicted_demand"]
        system_state["last_prediction"] = pred_val
        
        avail = system_state["available_generation"]
        shortage = round(max(0.0, pred_val - avail), 1)
        surplus = round(max(0.0, avail - pred_val), 1)

        # Status categorization
        if shortage > 0:
            status_text = "Deficit Expected"
            alert_type = "warning"
            alert_msg = f"Predicted demand ({pred_val} kW) exceeds available generation ({avail} kW). Shortage: {shortage} kW."
        else:
            status_text = "Balanced"
            alert_type = "success"
            alert_msg = f"Available generation ({avail} kW) covers predicted demand ({pred_val} kW). Surplus: {surplus} kW."

        response_payload = {
            "current_demand": system_state["current_demand"],
            "predicted_demand": pred_val,
            "available_generation": avail,
            "shortage": shortage,
            "surplus": surplus,
            "status": status_text,
            "alert_type": alert_type,
            "alert_message": alert_msg,
            "historical_trend": prediction_result["historical_trend"],
            "timeline_labels": prediction_result["timeline_labels"],
            "metrics": prediction_result["model_metrics"],
            "input_echo": prediction_result["input_echo"]
        }
        return jsonify(response_payload), 200

    except Exception as e:
        return jsonify({"error": f"Prediction failed: {str(e)}"}), 400


@app.route("/allocate", methods=["POST"])
def allocate():
    """
    POST /allocate
    Allocates available generation across load priorities (Critical -> Important -> Flexible).
    """
    try:
        data = request.get_json(force=True) or {}
        
        # Allow caller to override or use system state
        pred_demand = float(data.get("predicted_demand", system_state["last_prediction"]))
        avail_gen = float(data.get("available_generation", system_state["available_generation"]))
        loads = data.get("loads", system_state["loads"])

        allocation_result = allocate_energy(
            available_generation=avail_gen,
            predicted_demand=pred_demand,
            loads=loads
        )

        # Update cache
        system_state["last_allocation"] = allocation_result
        system_state["loads"] = allocation_result["loads"]
        system_state["available_generation"] = avail_gen
        system_state["last_prediction"] = pred_demand

        return jsonify(allocation_result), 200

    except Exception as e:
        return jsonify({"error": f"Allocation calculation failed: {str(e)}"}), 400


@app.route("/simulate/demand-surge", methods=["POST"])
def simulate_demand_surge():
    """
    POST /simulate/demand-surge
    Simulates a sudden demand spike (e.g. from 80 kW to 125 kW).
    Detects shortage, recalculates allocation, protects critical/important loads,
    and shifts flexible loads.
    """
    try:
        data = request.get_json(silent=True) or {}
        
        # Surge parameters
        surge_demand = float(data.get("surge_demand", 125.0))
        avail_gen = float(data.get("available_generation", 100.0))

        system_state["current_demand"] = surge_demand
        system_state["available_generation"] = avail_gen
        system_state["user_activity"] = "High"

        # Re-run ML prediction with surge inputs
        prediction_result = forecaster.predict({
            "current_demand": surge_demand,
            "temperature": max(34.0, system_state["temperature"]),
            "humidity": system_state["humidity"],
            "occupancy": 90.0,
            "hour": system_state["hour"],
            "day_type": system_state["day_type"],
            "user_activity": "High"
        })

        pred_demand = prediction_result["predicted_demand"]
        system_state["last_prediction"] = pred_demand

        # Run dynamic allocation
        allocation_result = allocate_energy(
            available_generation=avail_gen,
            predicted_demand=pred_demand,
            loads=get_default_loads()
        )

        shortage = round(max(0.0, pred_demand - avail_gen), 1)
        alert_msg = "Demand Surge Detected — Available energy is insufficient for current demand. Flexible loads are being managed."

        system_state["last_allocation"] = allocation_result
        system_state["loads"] = allocation_result["loads"]
        system_state["last_event"] = alert_msg

        return jsonify({
            "simulation": "Demand Surge",
            "current_demand": surge_demand,
            "predicted_demand": pred_demand,
            "available_generation": avail_gen,
            "shortage": shortage,
            "surplus": 0.0,
            "allocation": allocation_result,
            "historical_trend": prediction_result["historical_trend"],
            "timeline_labels": prediction_result["timeline_labels"],
            "alert_type": "danger",
            "alert_title": "Demand Surge Detected",
            "alert_message": alert_msg
        }), 200

    except Exception as e:
        return jsonify({"error": f"Demand surge simulation failed: {str(e)}"}), 500


@app.route("/simulate/generation-drop", methods=["POST"])
def simulate_generation_drop():
    """
    POST /simulate/generation-drop
    Simulates a sudden generation drop (e.g. from 100 kW to 70 kW due to cloud cover or trip).
    Detects generation deficit, recalculates allocation, protects priority loads,
    and manages flexible loads.
    """
    try:
        data = request.get_json(silent=True) or {}

        dropped_gen = float(data.get("dropped_generation", 70.0))
        current_demand = float(data.get("current_demand", 80.0))
        target_demand = float(data.get("predicted_demand", 100.0))

        system_state["available_generation"] = dropped_gen
        system_state["current_demand"] = current_demand
        system_state["last_prediction"] = target_demand

        # Run dynamic allocation with reduced generation
        allocation_result = allocate_energy(
            available_generation=dropped_gen,
            predicted_demand=target_demand,
            loads=get_default_loads()
        )

        shortage = round(max(0.0, target_demand - dropped_gen), 1)
        alert_msg = f"Generation Drop Detected — Available Energy: {dropped_gen} kW | Required Energy: {target_demand} kW | Shortage: {shortage} kW. Flexible loads throttled."

        system_state["last_allocation"] = allocation_result
        system_state["loads"] = allocation_result["loads"]
        system_state["last_event"] = alert_msg

        # Trend for visualization
        pred_res = forecaster.predict({
            "current_demand": current_demand,
            "temperature": system_state["temperature"],
            "humidity": system_state["humidity"],
            "occupancy": system_state["occupancy"],
            "hour": system_state["hour"],
            "day_type": system_state["day_type"],
            "user_activity": system_state["user_activity"]
        })

        return jsonify({
            "simulation": "Generation Drop",
            "current_demand": current_demand,
            "predicted_demand": target_demand,
            "available_generation": dropped_gen,
            "shortage": shortage,
            "surplus": 0.0,
            "allocation": allocation_result,
            "historical_trend": pred_res["historical_trend"],
            "timeline_labels": pred_res["timeline_labels"],
            "alert_type": "warning",
            "alert_title": "Generation Drop Detected",
            "alert_message": alert_msg
        }), 200

    except Exception as e:
        return jsonify({"error": f"Generation drop simulation failed: {str(e)}"}), 500


@app.route("/status", methods=["GET"])
def get_status():
    """
    GET /status
    Returns current telemetry, operational state, load allocation, and model health.
    """
    try:
        return jsonify({
            "system_state": {
                "current_demand": system_state["current_demand"],
                "available_generation": system_state["available_generation"],
                "temperature": system_state["temperature"],
                "humidity": system_state["humidity"],
                "occupancy": system_state["occupancy"],
                "hour": system_state["hour"],
                "day_type": system_state["day_type"],
                "user_activity": system_state["user_activity"],
                "last_prediction": system_state["last_prediction"],
                "last_event": system_state["last_event"]
            },
            "last_allocation": system_state["last_allocation"],
            "model_metrics": forecaster.metrics
        }), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/loads", methods=["GET"])
def get_loads():
    """
    GET /loads
    Returns the initial default categorized loads.
    """
    return jsonify({
        "loads": get_default_loads()
    }), 200


@app.route("/reset", methods=["POST"])
def reset():
    """
    POST /reset
    Restores baseline default inputs and allocation.
    """
    system_state["current_demand"] = 80.0
    system_state["available_generation"] = 100.0
    system_state["temperature"] = 32.0
    system_state["humidity"] = 60.0
    system_state["occupancy"] = 75.0
    system_state["hour"] = 14
    system_state["day_type"] = "Weekday"
    system_state["user_activity"] = "Medium"
    system_state["last_prediction"] = 94.0
    system_state["loads"] = get_default_loads()
    system_state["last_event"] = "System reset to baseline operation."

    alloc = allocate_energy(
        available_generation=100.0,
        predicted_demand=94.0,
        loads=system_state["loads"]
    )
    system_state["last_allocation"] = alloc

    pred_res = forecaster.predict({
        "current_demand": 80.0,
        "temperature": 32.0,
        "humidity": 60.0,
        "occupancy": 75.0,
        "hour": 14,
        "day_type": 0,
        "user_activity": 1
    })

    return jsonify({
        "message": "System reset to baseline successfully.",
        "system_state": system_state,
        "allocation": alloc,
        "historical_trend": pred_res["historical_trend"],
        "timeline_labels": pred_res["timeline_labels"]
    }), 200


if __name__ == "__main__":
    port = int(os.environ.get("PORT", 5000))
    print(f"Starting AI Smart Energy Management System on port {port}...")
    app.run(host="0.0.0.0", port=port, debug=True)

---
## 8. Frontend Dashboard Code

### 8.1 HTML Dashboard (`templates/index.html`)

In [ ]:
# HTML Template Code (Stored in templates/index.html)
HTML_CONTENT = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>AI Smart Energy Management System</title>
  <link rel="stylesheet" href="{{ url_for('static', filename='style.css') }}">
  <!-- Chart.js for data visualizations -->
  <script src="https://cdn.jsdelivr.net/npm/chart.js"></script>
</head>
<body>
  <!-- Header / Navigation Bar -->
  <header class="app-header">
    <div class="header-container">
      <div class="brand">
        <div class="brand-icon">
          <svg viewBox="0 0 24 24" width="24" height="24" stroke="currentColor" stroke-width="2" fill="none">
            <polygon points="13 2 3 14 12 14 11 22 21 10 12 10 13 2"></polygon>
          </svg>
        </div>
        <div>
          <h1 class="brand-title">AI Smart Energy Management System</h1>
          <p class="brand-subtitle">Demand Forecasting & Dynamic Energy Allocation</p>
        </div>
      </div>
      
      <div class="header-badges">
        <span class="badge badge-demo">⚡ Hackathon Prototype</span>
        <span class="badge badge-synthetic" title="Synthetic electricity dataset calibrated for hackathon validation">Synthetic ML Data</span>
        <div class="connection-status">
          <span class="status-dot pulse-green" id="connection-dot"></span>
          <span id="connection-text">Backend Active</span>
        </div>
      </div>
    </div>
  </header>

  <main class="dashboard-container">
    <!-- System Notification Banner -->
    <section class="banner-section">
      <div id="alert-banner" class="alert alert-success">
        <div class="alert-icon" id="alert-icon">✓</div>
        <div class="alert-content">
          <strong id="alert-title">System Status:</strong>
          <span id="alert-message">System operating normally. All critical and flexible loads satisfied.</span>
        </div>
        <div class="alert-time" id="alert-timestamp">Just now</div>
      </div>
    </section>

    <!-- Key Metrics Summary Cards -->
    <section class="metrics-grid">
      <!-- Card 1: Current Demand -->
      <div class="metric-card">
        <div class="card-header">
          <span class="card-label">Current Demand</span>
          <span class="card-badge">Real-time</span>
        </div>
        <div class="card-body">
          <span class="metric-value" id="val-current-demand">80.0</span>
          <span class="metric-unit">kW</span>
        </div>
        <div class="card-footer">Base historical operational demand</div>
      </div>

      <!-- Card 2: Predicted Demand -->
      <div class="metric-card highlight-purple">
        <div class="card-header">
          <span class="card-label">Predicted Demand</span>
          <span class="card-badge badge-ai">RandomForest ML</span>
        </div>
        <div class="card-body">
          <span class="metric-value" id="val-predicted-demand">94.0</span>
          <span class="metric-unit">kW</span>
        </div>
        <div class="card-footer" id="val-pred-delta">+14.0 kW from current load</div>
      </div>

      <!-- Card 3: Available Generation -->
      <div class="metric-card">
        <div class="card-header">
          <span class="card-label">Available Generation</span>
          <span class="card-badge badge-green">Supply</span>
        </div>
        <div class="card-body">
          <span class="metric-value" id="val-available-generation">100.0</span>
          <span class="metric-unit">kW</span>
        </div>
        <div class="card-footer">Grid capacity & on-site renewables</div>
      </div>

      <!-- Card 4: Shortage / Surplus -->
      <div class="metric-card" id="card-balance">
        <div class="card-header">
          <span class="card-label" id="label-balance">Shortage / Surplus</span>
          <span class="card-badge" id="badge-balance">Surplus</span>
        </div>
        <div class="card-body">
          <span class="metric-value" id="val-balance">6.0</span>
          <span class="metric-unit">kW</span>
        </div>
        <div class="card-footer" id="footer-balance">Grid capacity buffer available</div>
      </div>

      <!-- Card 5: Estimated Cost -->
      <div class="metric-card">
        <div class="card-header">
          <span class="card-label">Estimated Cost</span>
          <span class="card-badge badge-savings" id="badge-savings">Save $4.32</span>
        </div>
        <div class="card-body">
          <span class="metric-currency">$</span>
          <span class="metric-value" id="val-cost">15.04</span>
          <span class="metric-unit">/hr</span>
        </div>
        <div class="card-footer" id="footer-cost">Shifted flexible load savings applied</div>
      </div>

      <!-- Card 6: System Status -->
      <div class="metric-card">
        <div class="card-header">
          <span class="card-label">System Status</span>
          <span class="card-badge" id="badge-status">Active</span>
        </div>
        <div class="card-body status-container">
          <span class="status-indicator-dot" id="system-dot"></span>
          <span class="metric-status-text" id="val-system-status">Normal</span>
        </div>
        <div class="card-footer" id="footer-status">Protected: 100% Critical loads</div>
      </div>
    </section>

    <!-- Main Workspace Layout: Two Columns (Controls + Visualizations) -->
    <div class="workspace-grid">
      
      <!-- Left Column: Controls, Inputs & Scenarios -->
      <div class="controls-panel">
        
        <!-- Predefined Demo Scenarios -->
        <div class="panel-section">
          <div class="section-title-wrap">
            <h2 class="section-title">Demo Scenarios</h2>
            <span class="section-subtitle">1-Click Judges Test Flows</span>
          </div>
          <div class="scenario-buttons">
            <button class="btn btn-scenario" onclick="applyScenario(1)" title="Baseline balanced system">
              <span class="scenario-num">1</span>
              <span>Normal Demand</span>
            </button>
            <button class="btn btn-scenario" onclick="applyScenario(2)" title="High office activity and afternoon warmth">
              <span class="scenario-num">2</span>
              <span>High Demand</span>
            </button>
            <button class="btn btn-scenario btn-scenario-alert" onclick="applyScenario(3)" title="Sudden commercial/industrial surge">
              <span class="scenario-num">3</span>
              <span>Demand Surge (125 kW)</span>
            </button>
            <button class="btn btn-scenario btn-scenario-warning" onclick="applyScenario(4)" title="Sudden generation capacity drop">
              <span class="scenario-num">4</span>
              <span>Generation Drop (70 kW)</span>
            </button>
            <button class="btn btn-scenario" onclick="applyScenario(5)" title="Extreme heat wave & full occupancy">
              <span class="scenario-num">5</span>
              <span>Heatwave & High Occ</span>
            </button>
          </div>
        </div>

        <!-- Telemetry & Operating Parameters Form -->
        <div class="panel-section">
          <div class="section-title-wrap">
            <h2 class="section-title">Operating Parameters</h2>
            <span class="section-subtitle">Real-time Environmental & Load Data</span>
          </div>
          
          <form id="parameters-form" onsubmit="event.preventDefault(); runPrediction();">
            <div class="form-grid">
              
              <div class="form-group">
                <label for="input-current-demand">Current Demand (kW)</label>
                <div class="input-wrap">
                  <input type="number" id="input-current-demand" step="0.1" min="10" max="250" value="80.0" required>
                  <span class="unit-tag">kW</span>
                </div>
              </div>

              <div class="form-group">
                <label for="input-available-generation">Available Generation (kW)</label>
                <div class="input-wrap">
                  <input type="number" id="input-available-generation" step="0.1" min="10" max="300" value="100.0" required>
                  <span class="unit-tag">kW</span>
                </div>
              </div>

              <div class="form-group">
                <label for="input-temperature">Temperature (°C)</label>
                <div class="input-wrap">
                  <input type="number" id="input-temperature" step="0.5" min="10" max="50" value="32.0" required>
                  <span class="unit-tag">°C</span>
                </div>
              </div>

              <div class="form-group">
                <label for="input-humidity">Humidity (%)</label>
                <div class="input-wrap">
                  <input type="number" id="input-humidity" step="1" min="10" max="100" value="60" required>
                  <span class="unit-tag">%</span>
                </div>
              </div>

              <div class="form-group">
                <label for="input-occupancy">Occupancy (%)</label>
                <div class="input-wrap">
                  <input type="number" id="input-occupancy" step="1" min="0" max="100" value="75" required>
                  <span class="unit-tag">%</span>
                </div>
              </div>

              <div class="form-group">
                <label for="input-hour">Hour of Day (0 - 23)</label>
                <div class="input-wrap">
                  <input type="number" id="input-hour" min="0" max="23" value="14" required>
                  <span class="unit-tag">:00</span>
                </div>
              </div>

              <div class="form-group">
                <label for="input-day-type">Day Type</label>
                <select id="input-day-type">
                  <option value="Weekday" selected>Weekday (Mon-Fri)</option>
                  <option value="Weekend">Weekend (Sat-Sun)</option>
                </select>
              </div>

              <div class="form-group">
                <label for="input-user-activity">User Activity Level</label>
                <select id="input-user-activity">
                  <option value="Low">Low Activity</option>
                  <option value="Medium" selected>Medium Activity</option>
                  <option value="High">High Activity</option>
                </select>
              </div>

            </div>

            <!-- Primary Action Buttons -->
            <div class="action-buttons-group">
              <button type="button" class="btn btn-primary" onclick="runPrediction()">
                <svg viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2">
                  <path d="M21 16V8a2 2 0 0 0-1-1.73l-7-4a2 2 0 0 0-2 0l-7 4A2 2 0 0 0 3 8v8a2 2 0 0 0 1 1.73l7 4a2 2 0 0 0 2 0l7-4A2 2 0 0 0 21 16z"></path>
                  <polyline points="3.27 6.96 12 12.01 20.73 6.96"></polyline>
                  <line x1="12" y1="22.08" x2="12" y2="12"></line>
                </svg>
                <span>Run Prediction</span>
              </button>

              <button type="button" class="btn btn-secondary" onclick="allocateEnergy()">
                <svg viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2">
                  <circle cx="12" cy="12" r="3"></circle>
                  <path d="M19.4 15a1.65 1.65 0 0 0 .33 1.82l.06.06a2 2 0 0 1 0 2.83 2 2 0 0 1-2.83 0l-.06-.06a1.65 1.65 0 0 0-1.82-.33 1.65 1.65 0 0 0-1 1.51V21a2 2 0 0 1-2 2 2 2 0 0 1-2-2v-.09A1.65 1.65 0 0 0 9 19.4a1.65 1.65 0 0 0-1.82.33l-.06.06a2 2 0 0 1-2.83 0 2 2 0 0 1 0-2.83l.06-.06a1.65 1.65 0 0 0 .33-1.82 1.65 1.65 0 0 0-1.51-1H3a2 2 0 0 1-2-2 2 2 0 0 1 2-2h.09A1.65 1.65 0 0 0 4.6 9a1.65 1.65 0 0 0-.33-1.82l-.06-.06a2 2 0 0 1 0-2.83 2 2 0 0 1 2.83 0l.06.06a1.65 1.65 0 0 0 1.82.33H9a1.65 1.65 0 0 0 1-1.51V3a2 2 0 0 1 2-2 2 2 0 0 1 2 2v.09a1.65 1.65 0 0 0 1 1.51 1.65 1.65 0 0 0 1.82-.33l.06-.06a2 2 0 0 1 2.83 0 2 2 0 0 1 0 2.83l-.06.06a1.65 1.65 0 0 0-.33 1.82V9a1.65 1.65 0 0 0 1.51 1H21a2 2 0 0 1 2 2 2 2 0 0 1-2 2h-.09a1.65 1.65 0 0 0-1.51 1z"></path>
                </svg>
                <span>Allocate Energy</span>
              </button>

              <button type="button" class="btn btn-danger" onclick="simulateDemandSurge()">
                <svg viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2">
                  <polyline points="23 6 13.5 15.5 8.5 10.5 1 18"></polyline>
                  <polyline points="17 6 23 6 23 12"></polyline>
                </svg>
                <span>Simulate Demand Surge</span>
              </button>

              <button type="button" class="btn btn-warning" onclick="simulateGenerationDrop()">
                <svg viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2">
                  <polyline points="23 18 13.5 8.5 8.5 13.5 1 6"></polyline>
                  <polyline points="17 18 23 18 23 12"></polyline>
                </svg>
                <span>Simulate Generation Drop</span>
              </button>

              <button type="button" class="btn btn-outline" onclick="resetSystem()">
                <svg viewBox="0 0 24 24" width="18" height="18" fill="none" stroke="currentColor" stroke-width="2">
                  <polyline points="1 4 1 10 7 10"></polyline>
                  <path d="M3.51 15a9 9 0 1 0 2.13-9.36L1 10"></path>
                </svg>
                <span>Reset</span>
              </button>
            </div>
          </form>
        </div>

        <!-- ML Model Performance Badge -->
        <div class="panel-section model-meta-box">
          <div class="meta-row">
            <span class="meta-title">ML Model Architecture</span>
            <span class="meta-val">RandomForestRegressor (100 Trees)</span>
          </div>
          <div class="meta-row">
            <span class="meta-title">Mean Absolute Error (MAE)</span>
            <span class="meta-val" id="meta-mae">~3.27 kW</span>
          </div>
          <div class="meta-row">
            <span class="meta-title">R² Accuracy Score</span>
            <span class="meta-val" id="meta-r2">0.9617</span>
          </div>
          <div class="meta-row">
            <span class="meta-title">Validation Protocol</span>
            <span class="meta-val">80/20 Train/Test Split (2,000 samples)</span>
          </div>
        </div>

      </div>

      <!-- Right Column: Visualizations & Charts -->
      <div class="visuals-panel">
        
        <!-- Chart 1: Demand Forecast Line Chart -->
        <div class="chart-card">
          <div class="chart-card-header">
            <div>
              <h3 class="chart-title">Demand Forecast Timeline</h3>
              <p class="chart-subtitle">Historical Load Readings + AI Short-term Forecast Point</p>
            </div>
            <div class="legend-pills">
              <span class="pill pill-hist">● Historical</span>
              <span class="pill pill-pred">● AI Predicted (T+1h)</span>
            </div>
          </div>
          <div class="chart-wrapper">
            <canvas id="chart-demand-forecast"></canvas>
          </div>
        </div>

        <!-- Charts Row: 2 charts side by side -->
        <div class="charts-row">
          
          <!-- Chart 2: Demand vs Generation -->
          <div class="chart-card half-card">
            <div class="chart-card-header">
              <div>
                <h3 class="chart-title">Demand vs Generation</h3>
                <p class="chart-subtitle">Predicted Demand vs Available Supply</p>
              </div>
            </div>
            <div class="chart-wrapper small-chart">
              <canvas id="chart-demand-vs-generation"></canvas>
            </div>
          </div>

          <!-- Chart 3: Energy Allocation Breakdown -->
          <div class="chart-card half-card">
            <div class="chart-card-header">
              <div>
                <h3 class="chart-title">Dynamic Allocation Mix</h3>
                <p class="chart-subtitle">Critical vs Important vs Flexible kW</p>
              </div>
            </div>
            <div class="chart-wrapper small-chart">
              <canvas id="chart-energy-allocation"></canvas>
            </div>
          </div>

        </div>

        <!-- Cost & Savings Card -->
        <div class="cost-summary-box">
          <div class="cost-box-item">
            <span class="cost-box-label">Current Unoptimized Cost</span>
            <span class="cost-box-value" id="cost-unoptimized">$19.20/hr</span>
            <span class="cost-box-sub">Grid peak surcharge risk</span>
          </div>
          <div class="cost-divider">→</div>
          <div class="cost-box-item">
            <span class="cost-box-label">Optimized Smart Cost</span>
            <span class="cost-box-value highlight-green" id="cost-optimized">$14.88/hr</span>
            <span class="cost-box-sub">Shifted load off-peak tariff</span>
          </div>
          <div class="cost-divider">=</div>
          <div class="cost-box-item highlight-card">
            <span class="cost-box-label">Estimated Savings</span>
            <span class="cost-box-value saving-text" id="cost-savings">$4.32/hr</span>
            <span class="cost-box-sub" id="cost-savings-sub">~22.5% energy cost reduction</span>
          </div>
        </div>

      </div>

    </div>

    <!-- Bottom Section: Dynamic Load Classification & Status Table -->
    <section class="table-section">
      <div class="table-header-wrap">
        <div>
          <h2 class="section-title">Priority-Based Dynamic Load Allocation Table</h2>
          <p class="section-subtitle">Real-time status of Critical (Priority 1), Important (Priority 2), and Flexible (Priority 3) loads</p>
        </div>
        <div class="table-summary-badges">
          <span class="badge badge-critical" id="badge-critical-total">Critical: 45.0 kW (100%)</span>
          <span class="badge badge-important" id="badge-important-total">Important: 40.0 kW (100%)</span>
          <span class="badge badge-flexible" id="badge-flexible-total">Flexible: 15.0 kW (Managed)</span>
        </div>
      </div>

      <div class="table-responsive">
        <table class="load-table" id="loads-table">
          <thead>
            <tr>
              <th>Load Name</th>
              <th>Category</th>
              <th>Priority</th>
              <th>Required (kW)</th>
              <th>Allocated (kW)</th>
              <th>Fulfillment</th>
              <th>Status</th>
            </tr>
          </thead>
          <tbody id="loads-table-body">
            <!-- Dynamic rows populated via JavaScript -->
          </tbody>
        </table>
      </div>
    </section>

    <!-- Hackathon Methodology & Judge Reference Banner -->
    <footer class="app-footer">
      <div class="footer-note">
        <strong>Hackathon Demonstration Note:</strong>
        This system combines short-term <code>RandomForestRegressor</code> demand forecasting with multi-tier dynamic load shedding and time-of-use cost optimization. When demand surges or generation falls, flexible loads are intelligently throttled or deferred to eliminate blackout risk while preserving 100% supply to critical medical and server infrastructure.
      </div>
    </footer>
  </main>

  <script src="{{ url_for('static', filename='script.js') }}"></script>
</body>
</html>
"""
print('HTML template loaded. Characters:', len(HTML_CONTENT))

### 8.2 Modern CSS Theme (`static/style.css`)

In [ ]:
# CSS Stylesheet (Stored in static/style.css)
CSS_CONTENT = """/* ===================================================================
   AI Smart Energy Management System - Theme & Stylesheet
   Modern, Clean, High-Tech Hackathon UI for Judges & Demonstrations
   =================================================================== */

:root {
  --bg-main: #0a0e17;
  --bg-surface: #111827;
  --bg-surface-elevated: #1f2937;
  --bg-surface-glass: rgba(17, 24, 39, 0.75);
  
  --border-subtle: rgba(255, 255, 255, 0.08);
  --border-accent: rgba(56, 189, 248, 0.35);
  
  --text-primary: #f8fafc;
  --text-secondary: #94a3b8;
  --text-muted: #64748b;
  
  --accent-cyan: #38bdf8;
  --accent-blue: #3b82f6;
  --accent-purple: #a855f7;
  --accent-green: #10b981;
  --accent-amber: #f59e0b;
  --accent-red: #ef4444;
  
  --font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif;
  --radius-sm: 6px;
  --radius-md: 10px;
  --radius-lg: 14px;
  --radius-pill: 9999px;
  
  --shadow-sm: 0 1px 3px rgba(0, 0, 0, 0.3);
  --shadow-md: 0 4px 12px rgba(0, 0, 0, 0.4);
  --shadow-glow-cyan: 0 0 15px rgba(56, 189, 248, 0.25);
  --shadow-glow-green: 0 0 15px rgba(16, 185, 129, 0.25);
  --shadow-glow-red: 0 0 15px rgba(239, 68, 68, 0.3);
}

* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

body {
  background-color: var(--bg-main);
  color: var(--text-primary);
  font-family: var(--font-family);
  line-height: 1.5;
  min-height: 100vh;
  padding-bottom: 40px;
}

/* ================= HEADER ================= */
.app-header {
  background: rgba(10, 14, 23, 0.85);
  backdrop-filter: blur(12px);
  border-bottom: 1px solid var(--border-subtle);
  position: sticky;
  top: 0;
  z-index: 100;
  padding: 14px 24px;
}

.header-container {
  max-width: 1400px;
  margin: 0 auto;
  display: flex;
  justify-content: space-between;
  align-items: center;
  flex-wrap: wrap;
  gap: 16px;
}

.brand {
  display: flex;
  align-items: center;
  gap: 12px;
}

.brand-icon {
  width: 40px;
  height: 40px;
  background: linear-gradient(135deg, #0284c7, #38bdf8);
  border-radius: var(--radius-md);
  display: flex;
  align-items: center;
  justify-content: center;
  color: #fff;
  box-shadow: var(--shadow-glow-cyan);
}

.brand-title {
  font-size: 1.25rem;
  font-weight: 700;
  letter-spacing: -0.02em;
  color: #fff;
}

.brand-subtitle {
  font-size: 0.82rem;
  color: var(--text-secondary);
}

.header-badges {
  display: flex;
  align-items: center;
  gap: 10px;
}

.badge {
  font-size: 0.75rem;
  font-weight: 600;
  padding: 4px 10px;
  border-radius: var(--radius-pill);
  letter-spacing: 0.03em;
  text-transform: uppercase;
}

.badge-demo {
  background: rgba(56, 189, 248, 0.15);
  color: var(--accent-cyan);
  border: 1px solid rgba(56, 189, 248, 0.3);
}

.badge-synthetic {
  background: rgba(168, 85, 247, 0.15);
  color: var(--accent-purple);
  border: 1px solid rgba(168, 85, 247, 0.3);
}

.connection-status {
  display: flex;
  align-items: center;
  gap: 6px;
  font-size: 0.78rem;
  color: var(--text-secondary);
  background: var(--bg-surface);
  padding: 4px 10px;
  border-radius: var(--radius-pill);
  border: 1px solid var(--border-subtle);
}

.status-dot {
  width: 8px;
  height: 8px;
  border-radius: 50%;
  display: inline-block;
}

.pulse-green {
  background-color: var(--accent-green);
  box-shadow: 0 0 8px var(--accent-green);
}

/* ================= DASHBOARD MAIN ================= */
.dashboard-container {
  max-width: 1400px;
  margin: 20px auto;
  padding: 0 20px;
  display: flex;
  flex-direction: column;
  gap: 22px;
}

/* ================= SYSTEM NOTIFICATION BANNER ================= */
.alert {
  display: flex;
  align-items: center;
  gap: 14px;
  padding: 12px 18px;
  border-radius: var(--radius-md);
  font-size: 0.92rem;
  border: 1px solid transparent;
  transition: all 0.3s ease;
}

.alert-icon {
  font-size: 1.1rem;
  font-weight: bold;
}

.alert-content {
  flex: 1;
}

.alert-content strong {
  margin-right: 6px;
}

.alert-time {
  font-size: 0.76rem;
  color: var(--text-muted);
}

.alert-success {
  background: rgba(16, 185, 129, 0.1);
  border-color: rgba(16, 185, 129, 0.3);
  color: #34d399;
}

.alert-warning {
  background: rgba(245, 158, 11, 0.12);
  border-color: rgba(245, 158, 11, 0.4);
  color: #fbbf24;
}

.alert-danger {
  background: rgba(239, 68, 68, 0.14);
  border-color: rgba(239, 68, 68, 0.45);
  color: #f87171;
}

/* ================= METRIC CARDS ================= */
.metrics-grid {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
  gap: 14px;
}

.metric-card {
  background: var(--bg-surface);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-lg);
  padding: 16px 18px;
  display: flex;
  flex-direction: column;
  justify-content: space-between;
  box-shadow: var(--shadow-sm);
  transition: transform 0.2s ease, border-color 0.2s ease;
}

.metric-card:hover {
  border-color: rgba(255, 255, 255, 0.15);
  transform: translateY(-2px);
}

.metric-card.highlight-purple {
  border-color: rgba(168, 85, 247, 0.35);
  background: linear-gradient(180deg, rgba(168, 85, 247, 0.08) 0%, var(--bg-surface) 100%);
}

.card-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  margin-bottom: 8px;
}

.card-label {
  font-size: 0.8rem;
  font-weight: 600;
  text-transform: uppercase;
  letter-spacing: 0.04em;
  color: var(--text-secondary);
}

.card-badge {
  font-size: 0.68rem;
  padding: 2px 7px;
  border-radius: var(--radius-pill);
  background: var(--bg-surface-elevated);
  color: var(--text-muted);
  font-weight: 600;
}

.card-badge.badge-ai {
  background: rgba(168, 85, 247, 0.2);
  color: #c084fc;
}

.card-badge.badge-green {
  background: rgba(16, 185, 129, 0.2);
  color: #34d399;
}

.card-badge.badge-red {
  background: rgba(239, 68, 68, 0.2);
  color: #f87171;
}

.card-badge.badge-savings {
  background: rgba(16, 185, 129, 0.2);
  color: #34d399;
  font-weight: 700;
}

.card-body {
  display: flex;
  align-items: baseline;
  gap: 6px;
  margin-bottom: 6px;
}

.metric-value {
  font-size: 1.85rem;
  font-weight: 800;
  color: #fff;
  letter-spacing: -0.03em;
}

.metric-unit {
  font-size: 0.9rem;
  color: var(--text-muted);
  font-weight: 500;
}

.metric-currency {
  font-size: 1.3rem;
  color: var(--accent-green);
  font-weight: 700;
}

.card-footer {
  font-size: 0.74rem;
  color: var(--text-muted);
}

.status-container {
  display: flex;
  align-items: center;
  gap: 8px;
}

.status-indicator-dot {
  width: 10px;
  height: 10px;
  border-radius: 50%;
  background: var(--accent-green);
  box-shadow: 0 0 10px var(--accent-green);
}

.metric-status-text {
  font-size: 1.3rem;
  font-weight: 700;
}

/* ================= WORKSPACE TWO-COLUMN ================= */
.workspace-grid {
  display: grid;
  grid-template-columns: 460px 1fr;
  gap: 20px;
}

@media (max-width: 1100px) {
  .workspace-grid {
    grid-template-columns: 1fr;
  }
}

.controls-panel, .visuals-panel {
  display: flex;
  flex-direction: column;
  gap: 18px;
}

.panel-section {
  background: var(--bg-surface);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-lg);
  padding: 20px;
  box-shadow: var(--shadow-sm);
}

.section-title-wrap {
  margin-bottom: 16px;
}

.section-title {
  font-size: 1.05rem;
  font-weight: 700;
  color: #fff;
}

.section-subtitle {
  font-size: 0.78rem;
  color: var(--text-muted);
}

/* ================= DEMO SCENARIOS ================= */
.scenario-buttons {
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.btn-scenario {
  background: var(--bg-surface-elevated);
  border: 1px solid var(--border-subtle);
  color: var(--text-primary);
  padding: 9px 12px;
  border-radius: var(--radius-md);
  font-size: 0.84rem;
  font-weight: 600;
  cursor: pointer;
  display: flex;
  align-items: center;
  gap: 10px;
  text-align: left;
  transition: all 0.15s ease;
}

.btn-scenario:hover {
  background: #374151;
  border-color: rgba(255, 255, 255, 0.2);
  transform: translateX(3px);
}

.scenario-num {
  background: rgba(255, 255, 255, 0.1);
  color: #fff;
  width: 22px;
  height: 22px;
  border-radius: 50%;
  display: flex;
  align-items: center;
  justify-content: center;
  font-size: 0.75rem;
  font-weight: 700;
  flex-shrink: 0;
}

.btn-scenario-alert {
  border-left: 3px solid var(--accent-red);
}

.btn-scenario-warning {
  border-left: 3px solid var(--accent-amber);
}

/* ================= FORM CONTROLS ================= */
.form-grid {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 12px;
  margin-bottom: 18px;
}

.form-group {
  display: flex;
  flex-direction: column;
  gap: 4px;
}

.form-group label {
  font-size: 0.76rem;
  font-weight: 600;
  color: var(--text-secondary);
}

.input-wrap {
  position: relative;
  display: flex;
  align-items: center;
}

.input-wrap input,
select {
  width: 100%;
  background: #0f172a;
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-sm);
  padding: 8px 34px 8px 10px;
  color: #fff;
  font-size: 0.88rem;
  font-family: inherit;
  transition: border-color 0.2s ease, box-shadow 0.2s ease;
}

select {
  padding-right: 12px;
}

.input-wrap input:focus,
select:focus {
  outline: none;
  border-color: var(--accent-cyan);
  box-shadow: 0 0 0 2px rgba(56, 189, 248, 0.2);
}

.unit-tag {
  position: absolute;
  right: 10px;
  font-size: 0.75rem;
  color: var(--text-muted);
  pointer-events: none;
}

/* ================= ACTION BUTTONS ================= */
.action-buttons-group {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 10px;
}

.btn {
  display: flex;
  align-items: center;
  justify-content: center;
  gap: 8px;
  padding: 10px 14px;
  border-radius: var(--radius-md);
  font-size: 0.84rem;
  font-weight: 600;
  cursor: pointer;
  border: 1px solid transparent;
  transition: all 0.2s ease;
}

.btn:active {
  transform: scale(0.98);
}

.btn-primary {
  background: linear-gradient(135deg, #0284c7, #0ea5e9);
  color: #fff;
  box-shadow: var(--shadow-glow-cyan);
}

.btn-primary:hover {
  background: linear-gradient(135deg, #0369a1, #0284c7);
}

.btn-secondary {
  background: linear-gradient(135deg, #059669, #10b981);
  color: #fff;
  box-shadow: var(--shadow-glow-green);
}

.btn-secondary:hover {
  background: linear-gradient(135deg, #047857, #059669);
}

.btn-danger {
  background: linear-gradient(135deg, #b91c1c, #dc2626);
  color: #fff;
  box-shadow: var(--shadow-glow-red);
}

.btn-danger:hover {
  background: linear-gradient(135deg, #991b1b, #b91c1c);
}

.btn-warning {
  background: linear-gradient(135deg, #d97706, #f59e0b);
  color: #111;
  font-weight: 700;
}

.btn-warning:hover {
  background: linear-gradient(135deg, #b45309, #d97706);
}

.btn-outline {
  grid-column: span 2;
  background: transparent;
  border-color: var(--border-subtle);
  color: var(--text-secondary);
}

.btn-outline:hover {
  background: rgba(255, 255, 255, 0.05);
  color: #fff;
}

/* ================= MODEL META BOX ================= */
.model-meta-box {
  background: rgba(17, 24, 39, 0.6);
  border: 1px dashed rgba(255, 255, 255, 0.12);
  display: flex;
  flex-direction: column;
  gap: 8px;
}

.meta-row {
  display: flex;
  justify-content: space-between;
  align-items: center;
  font-size: 0.78rem;
}

.meta-title {
  color: var(--text-muted);
}

.meta-val {
  font-weight: 600;
  color: var(--text-primary);
}

/* ================= CHARTS ================= */
.chart-card {
  background: var(--bg-surface);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-lg);
  padding: 18px 20px;
  box-shadow: var(--shadow-sm);
}

.chart-card-header {
  display: flex;
  justify-content: space-between;
  align-items: center;
  margin-bottom: 14px;
}

.chart-title {
  font-size: 0.98rem;
  font-weight: 700;
  color: #fff;
}

.chart-subtitle {
  font-size: 0.76rem;
  color: var(--text-muted);
}

.legend-pills {
  display: flex;
  gap: 12px;
  font-size: 0.75rem;
}

.pill-hist {
  color: #38bdf8;
}

.pill-pred {
  color: #c084fc;
}

.chart-wrapper {
  position: relative;
  width: 100%;
  height: 220px;
}

.charts-row {
  display: grid;
  grid-template-columns: 1fr 1fr;
  gap: 18px;
}

@media (max-width: 768px) {
  .charts-row {
    grid-template-columns: 1fr;
  }
}

.chart-wrapper.small-chart {
  height: 190px;
}

/* ================= COST & SAVINGS BOX ================= */
.cost-summary-box {
  background: var(--bg-surface);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-lg);
  padding: 16px 22px;
  display: flex;
  align-items: center;
  justify-content: space-around;
  flex-wrap: wrap;
  gap: 14px;
}

.cost-box-item {
  display: flex;
  flex-direction: column;
  gap: 2px;
}

.cost-box-label {
  font-size: 0.74rem;
  font-weight: 600;
  color: var(--text-muted);
  text-transform: uppercase;
}

.cost-box-value {
  font-size: 1.45rem;
  font-weight: 800;
  color: #fff;
}

.cost-box-sub {
  font-size: 0.72rem;
  color: var(--text-muted);
}

.cost-divider {
  font-size: 1.2rem;
  color: var(--text-muted);
}

.highlight-green {
  color: var(--accent-green);
}

.saving-text {
  color: #34d399;
}

.highlight-card {
  background: rgba(16, 185, 129, 0.1);
  border: 1px solid rgba(16, 185, 129, 0.25);
  padding: 8px 14px;
  border-radius: var(--radius-md);
}

/* ================= LOADS TABLE ================= */
.table-section {
  background: var(--bg-surface);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-lg);
  padding: 22px;
  box-shadow: var(--shadow-sm);
}

.table-header-wrap {
  display: flex;
  justify-content: space-between;
  align-items: center;
  flex-wrap: wrap;
  gap: 12px;
  margin-bottom: 16px;
}

.table-summary-badges {
  display: flex;
  gap: 10px;
  flex-wrap: wrap;
}

.badge-critical {
  background: rgba(239, 68, 68, 0.15);
  color: #f87171;
  border: 1px solid rgba(239, 68, 68, 0.3);
}

.badge-important {
  background: rgba(245, 158, 11, 0.15);
  color: #fbbf24;
  border: 1px solid rgba(245, 158, 11, 0.3);
}

.badge-flexible {
  background: rgba(56, 189, 248, 0.15);
  color: #38bdf8;
  border: 1px solid rgba(56, 189, 248, 0.3);
}

.table-responsive {
  overflow-x: auto;
}

.load-table {
  width: 100%;
  border-collapse: collapse;
  font-size: 0.86rem;
  text-align: left;
}

.load-table th {
  background: #0f172a;
  color: var(--text-muted);
  font-weight: 600;
  text-transform: uppercase;
  font-size: 0.72rem;
  letter-spacing: 0.04em;
  padding: 10px 14px;
  border-bottom: 1px solid var(--border-subtle);
}

.load-table td {
  padding: 12px 14px;
  border-bottom: 1px solid rgba(255, 255, 255, 0.04);
  color: var(--text-primary);
}

.load-table tr:hover td {
  background: rgba(255, 255, 255, 0.02);
}

.table-badge {
  display: inline-block;
  font-size: 0.72rem;
  font-weight: 600;
  padding: 3px 8px;
  border-radius: var(--radius-sm);
}

.tag-critical {
  background: rgba(239, 68, 68, 0.2);
  color: #f87171;
}

.tag-important {
  background: rgba(245, 158, 11, 0.2);
  color: #fbbf24;
}

.tag-flexible {
  background: rgba(56, 189, 248, 0.2);
  color: #38bdf8;
}

.status-active {
  background: rgba(16, 185, 129, 0.2);
  color: #34d399;
}

.status-managed {
  background: rgba(245, 158, 11, 0.2);
  color: #fbbf24;
}

.status-shifted {
  background: rgba(168, 85, 247, 0.2);
  color: #c084fc;
}

.status-reduced {
  background: rgba(249, 115, 22, 0.2);
  color: #fb923c;
}

.progress-bar-bg {
  width: 90px;
  height: 6px;
  background: rgba(255, 255, 255, 0.1);
  border-radius: 4px;
  overflow: hidden;
  display: inline-block;
  vertical-align: middle;
  margin-right: 8px;
}

.progress-bar-fill {
  height: 100%;
  border-radius: 4px;
  transition: width 0.3s ease;
}

/* ================= FOOTER ================= */
.app-footer {
  margin-top: 10px;
}

.footer-note {
  background: rgba(15, 23, 42, 0.6);
  border: 1px solid var(--border-subtle);
  border-radius: var(--radius-md);
  padding: 12px 18px;
  font-size: 0.78rem;
  color: var(--text-muted);
  line-height: 1.5;
}

.footer-note code {
  background: rgba(255, 255, 255, 0.08);
  padding: 2px 5px;
  border-radius: 4px;
  color: var(--accent-cyan);
}
"""
print('CSS stylesheet loaded. Characters:', len(CSS_CONTENT))

### 8.3 JavaScript & Chart.js Controller (`static/script.js`)

In [ ]:
# JavaScript Controller (Stored in static/script.js)
JS_CONTENT = """/**
 * AI Smart Energy Management System - Frontend Controller
 * Communicates with Flask backend REST endpoints, animates Chart.js visualizations,
 * and handles interactive scenario simulations.
 */

let forecastChart = null;
let demandVsGenChart = null;
let allocationChart = null;

// Initialize on page load
document.addEventListener("DOMContentLoaded", () => {
  initCharts();
  fetchInitialStatus();
});

/* ================= CHART INITIALIZATION ================= */
function initCharts() {
  // Chart 1: Demand Forecast Timeline
  const ctxForecast = document.getElementById("chart-demand-forecast").getContext("2d");
  forecastChart = new Chart(ctxForecast, {
    type: "line",
    data: {
      labels: ["T-4h", "T-3h", "T-2h", "T-1h", "Current", "Forecast (T+1h)"],
      datasets: [
        {
          label: "Demand Curve (kW)",
          data: [68, 72, 75, 78, 80, 94],
          borderColor: "#38bdf8",
          backgroundColor: "rgba(56, 189, 248, 0.12)",
          fill: true,
          tension: 0.35,
          borderWidth: 3,
          pointBackgroundColor: ["#38bdf8", "#38bdf8", "#38bdf8", "#38bdf8", "#38bdf8", "#c084fc"],
          pointBorderColor: "#fff",
          pointRadius: [4, 4, 4, 4, 6, 8],
          pointHoverRadius: 9
        }
      ]
    },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        tooltip: {
          backgroundColor: "#1e293b",
          titleColor: "#f8fafc",
          bodyColor: "#38bdf8",
          borderColor: "rgba(255,255,255,0.1)",
          borderWidth: 1,
          padding: 10,
          callbacks: {
            label: (ctx) => `${ctx.dataset.label}: ${ctx.raw} kW`
          }
        }
      },
      scales: {
        x: {
          grid: { color: "rgba(255, 255, 255, 0.05)" },
          ticks: { color: "#94a3b8", font: { size: 11 } }
        },
        y: {
          grid: { color: "rgba(255, 255, 255, 0.05)" },
          ticks: {
            color: "#94a3b8",
            font: { size: 11 },
            callback: (v) => `${v} kW`
          }
        }
      }
    }
  });

  // Chart 2: Demand vs Generation Bar Chart
  const ctxCompare = document.getElementById("chart-demand-vs-generation").getContext("2d");
  demandVsGenChart = new Chart(ctxCompare, {
    type: "bar",
    data: {
      labels: ["Predicted Demand", "Available Generation"],
      datasets: [
        {
          label: "kW",
          data: [94, 100],
          backgroundColor: ["rgba(168, 85, 247, 0.8)", "rgba(16, 185, 129, 0.8)"],
          borderColor: ["#c084fc", "#34d399"],
          borderWidth: 1.5,
          borderRadius: 6,
          barPercentage: 0.55
        }
      ]
    },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      plugins: {
        legend: { display: false },
        tooltip: {
          backgroundColor: "#1e293b",
          padding: 8,
          callbacks: { label: (ctx) => `${ctx.raw} kW` }
        }
      },
      scales: {
        x: {
          grid: { display: false },
          ticks: { color: "#94a3b8", font: { size: 11 } }
        },
        y: {
          grid: { color: "rgba(255, 255, 255, 0.05)" },
          ticks: { color: "#94a3b8", font: { size: 11 } }
        }
      }
    }
  });

  // Chart 3: Energy Allocation Doughnut Chart
  const ctxAlloc = document.getElementById("chart-energy-allocation").getContext("2d");
  allocationChart = new Chart(ctxAlloc, {
    type: "doughnut",
    data: {
      labels: ["Critical Loads", "Important Loads", "Flexible Loads"],
      datasets: [
        {
          data: [45, 40, 9],
          backgroundColor: [
            "rgba(239, 68, 68, 0.85)",   // Red/Coral
            "rgba(245, 158, 11, 0.85)",  // Amber
            "rgba(56, 189, 248, 0.85)"   // Cyan
          ],
          borderColor: "#111827",
          borderWidth: 2,
          hoverOffset: 4
        }
      ]
    },
    options: {
      responsive: true,
      maintainAspectRatio: false,
      cutout: "68%",
      plugins: {
        legend: {
          position: "bottom",
          labels: {
            color: "#94a3b8",
            boxWidth: 12,
            font: { size: 11 },
            padding: 8
          }
        },
        tooltip: {
          backgroundColor: "#1e293b",
          padding: 8,
          callbacks: {
            label: (ctx) => ` ${ctx.label}: ${ctx.raw} kW`
          }
        }
      }
    }
  });
}

/* ================= INITIAL STATUS FETCH ================= */
async function fetchInitialStatus() {
  try {
    const res = await fetch("/status");
    if (!res.ok) throw new Error("Status endpoint not responding");
    const data = await res.json();
    
    if (data.system_state) {
      document.getElementById("input-current-demand").value = data.system_state.current_demand;
      document.getElementById("input-available-generation").value = data.system_state.available_generation;
      document.getElementById("input-temperature").value = data.system_state.temperature;
      document.getElementById("input-humidity").value = data.system_state.humidity;
      document.getElementById("input-occupancy").value = data.system_state.occupancy;
      document.getElementById("input-hour").value = data.system_state.hour;
      document.getElementById("input-day-type").value = data.system_state.day_type;
      document.getElementById("input-user-activity").value = data.system_state.user_activity;
      
      document.getElementById("val-current-demand").textContent = data.system_state.current_demand.toFixed(1);
      document.getElementById("val-predicted-demand").textContent = data.system_state.last_prediction.toFixed(1);
      document.getElementById("val-available-generation").textContent = data.system_state.available_generation.toFixed(1);
    }

    if (data.model_metrics) {
      if (data.model_metrics.mae) {
        document.getElementById("meta-mae").textContent = `${data.model_metrics.mae} kW`;
      }
      if (data.model_metrics.r2_score) {
        document.getElementById("meta-r2").textContent = data.model_metrics.r2_score;
      }
    }

    if (data.last_allocation) {
      updateAllocationUI(data.last_allocation);
    }
    
    setConnectionActive(true);
  } catch (err) {
    console.error("Initial load error:", err);
    setConnectionActive(false);
  }
}

/* ================= API CALL: RUN PREDICTION ================= */
async function runPrediction() {
  const payload = getFormPayload();
  showBannerLoading("Running AI demand forecasting model...");

  try {
    const res = await fetch("/predict", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify(payload)
    });
    
    if (!res.ok) {
      const err = await res.json();
      throw new Error(err.error || "Prediction failed");
    }

    const data = await res.json();

    // Update Top Metric Cards
    document.getElementById("val-current-demand").textContent = data.current_demand.toFixed(1);
    document.getElementById("val-predicted-demand").textContent = data.predicted_demand.toFixed(1);
    document.getElementById("val-available-generation").textContent = data.available_generation.toFixed(1);

    const delta = data.predicted_demand - data.current_demand;
    const deltaSign = delta >= 0 ? "+" : "";
    document.getElementById("val-pred-delta").textContent = `${deltaSign}${delta.toFixed(1)} kW from current load`;

    // Update Forecast Chart
    if (forecastChart && data.historical_trend) {
      const chartPoints = [...data.historical_trend, data.predicted_demand];
      forecastChart.data.labels = data.timeline_labels || ["T-4h", "T-3h", "T-2h", "T-1h", "Current", "Forecast (T+1h)"];
      forecastChart.data.datasets[0].data = chartPoints;
      forecastChart.update();
    }

    // Immediately trigger allocation with the newly predicted demand
    await allocateEnergy(data.predicted_demand, data.available_generation);

  } catch (err) {
    showAlert("danger", "Prediction Error", err.message);
  }
}

/* ================= API CALL: ALLOCATE ENERGY ================= */
async function allocateEnergy(overrideDemand = null, overrideGen = null) {
  const currentDemand = overrideDemand !== null ? overrideDemand : parseFloat(document.getElementById("val-predicted-demand").textContent);
  const availGen = overrideGen !== null ? overrideGen : parseFloat(document.getElementById("input-available-generation").value);

  try {
    const res = await fetch("/allocate", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({
        predicted_demand: currentDemand,
        available_generation: availGen
      })
    });

    if (!res.ok) {
      const err = await res.json();
      throw new Error(err.error || "Allocation failed");
    }

    const data = await res.json();
    updateAllocationUI(data);

  } catch (err) {
    showAlert("danger", "Allocation Error", err.message);
  }
}

/* ================= API CALL: DEMAND SURGE SIMULATION ================= */
async function simulateDemandSurge() {
  showBannerLoading("Simulating sudden high-demand spike...");
  
  try {
    const res = await fetch("/simulate/demand-surge", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({
        surge_demand: 125.0,
        available_generation: 100.0
      })
    });

    if (!res.ok) throw new Error("Demand surge simulation failed");
    const data = await res.json();

    // Update form values
    document.getElementById("input-current-demand").value = data.current_demand;
    document.getElementById("input-available-generation").value = data.available_generation;
    document.getElementById("input-user-activity").value = "High";

    // Update Top Metric Cards
    document.getElementById("val-current-demand").textContent = data.current_demand.toFixed(1);
    document.getElementById("val-predicted-demand").textContent = data.predicted_demand.toFixed(1);
    document.getElementById("val-available-generation").textContent = data.available_generation.toFixed(1);
    document.getElementById("val-pred-delta").textContent = "+25.0 kW sudden surge detected";

    // Update Chart
    if (forecastChart && data.historical_trend) {
      forecastChart.data.labels = data.timeline_labels;
      forecastChart.data.datasets[0].data = [...data.historical_trend, data.predicted_demand];
      forecastChart.update();
    }

    // Update Allocation and Alert
    updateAllocationUI(data.allocation);
    showAlert("danger", data.alert_title, data.alert_message);

  } catch (err) {
    showAlert("danger", "Surge Simulation Error", err.message);
  }
}

/* ================= API CALL: GENERATION DROP SIMULATION ================= */
async function simulateGenerationDrop() {
  showBannerLoading("Simulating generation loss / grid trip...");

  try {
    const res = await fetch("/simulate/generation-drop", {
      method: "POST",
      headers: { "Content-Type": "application/json" },
      body: JSON.stringify({
        dropped_generation: 70.0,
        predicted_demand: 100.0
      })
    });

    if (!res.ok) throw new Error("Generation drop simulation failed");
    const data = await res.json();

    // Update form values
    document.getElementById("input-available-generation").value = data.available_generation;

    // Update Top Metric Cards
    document.getElementById("val-available-generation").textContent = data.available_generation.toFixed(1);
    document.getElementById("val-predicted-demand").textContent = data.predicted_demand.toFixed(1);

    // Update Allocation and Alert
    updateAllocationUI(data.allocation);
    showAlert("warning", data.alert_title, data.alert_message);

  } catch (err) {
    showAlert("danger", "Generation Drop Error", err.message);
  }
}

/* ================= PREDEFINED DEMO SCENARIOS ================= */
async function applyScenario(num) {
  if (num === 3) {
    // Scenario 3: Sudden Demand Surge
    await simulateDemandSurge();
    return;
  }
  if (num === 4) {
    // Scenario 4: Generation Drop
    await simulateGenerationDrop();
    return;
  }

  // Define inputs for other scenarios
  let scenarioConfig = {};
  if (num === 1) {
    // Scenario 1: Normal Demand
    scenarioConfig = {
      current_demand: 80.0,
      available_generation: 100.0,
      temperature: 25.0,
      humidity: 50,
      occupancy: 50,
      hour: 10,
      day_type: "Weekday",
      user_activity: "Medium"
    };
  } else if (num === 2) {
    // Scenario 2: High Demand
    scenarioConfig = {
      current_demand: 95.0,
      available_generation: 90.0,
      temperature: 34.0,
      humidity: 65,
      occupancy: 85,
      hour: 14,
      day_type: "Weekday",
      user_activity: "High"
    };
  } else if (num === 5) {
    // Scenario 5: High Temperature + High Occupancy
    scenarioConfig = {
      current_demand: 100.0,
      available_generation: 95.0,
      temperature: 39.5,
      humidity: 78,
      occupancy: 98,
      hour: 15,
      day_type: "Weekday",
      user_activity: "High"
    };
  }

  // Populate form fields
  document.getElementById("input-current-demand").value = scenarioConfig.current_demand;
  document.getElementById("input-available-generation").value = scenarioConfig.available_generation;
  document.getElementById("input-temperature").value = scenarioConfig.temperature;
  document.getElementById("input-humidity").value = scenarioConfig.humidity;
  document.getElementById("input-occupancy").value = scenarioConfig.occupancy;
  document.getElementById("input-hour").value = scenarioConfig.hour;
  document.getElementById("input-day-type").value = scenarioConfig.day_type;
  document.getElementById("input-user-activity").value = scenarioConfig.user_activity;

  // Run full pipeline
  await runPrediction();
}

/* ================= SYSTEM RESET ================= */
async function resetSystem() {
  try {
    const res = await fetch("/reset", { method: "POST" });
    const data = await res.json();
    
    // Reset Form
    document.getElementById("input-current-demand").value = 80.0;
    document.getElementById("input-available-generation").value = 100.0;
    document.getElementById("input-temperature").value = 32.0;
    document.getElementById("input-humidity").value = 60;
    document.getElementById("input-occupancy").value = 75;
    document.getElementById("input-hour").value = 14;
    document.getElementById("input-day-type").value = "Weekday";
    document.getElementById("input-user-activity").value = "Medium";

    // Update Cards
    document.getElementById("val-current-demand").textContent = "80.0";
    document.getElementById("val-predicted-demand").textContent = "94.0";
    document.getElementById("val-available-generation").textContent = "100.0";
    document.getElementById("val-pred-delta").textContent = "+14.0 kW from current load";

    // Update Forecast Chart
    if (forecastChart && data.historical_trend) {
      forecastChart.data.labels = data.timeline_labels;
      forecastChart.data.datasets[0].data = [...data.historical_trend, 94.0];
      forecastChart.update();
    }

    updateAllocationUI(data.allocation);
    showAlert("success", "System Reset", "All operational parameters restored to baseline.");

  } catch (err) {
    showAlert("danger", "Reset Error", err.message);
  }
}

/* ================= UI RENDER HELPERS ================= */
function updateAllocationUI(alloc) {
  if (!alloc) return;

  // 1. Balance Card (Shortage / Surplus)
  const isShortage = alloc.shortage > 0;
  const labelBalance = document.getElementById("label-balance");
  const badgeBalance = document.getElementById("badge-balance");
  const valBalance = document.getElementById("val-balance");
  const footerBalance = document.getElementById("footer-balance");

  if (isShortage) {
    labelBalance.textContent = "Energy Shortage";
    badgeBalance.textContent = "Deficit";
    badgeBalance.className = "card-badge badge-red";
    valBalance.textContent = alloc.shortage.toFixed(1);
    valBalance.style.color = "#f87171";
    footerBalance.textContent = "Flexible loads throttled/deferred";
  } else {
    labelBalance.textContent = "Energy Surplus";
    badgeBalance.textContent = "Surplus";
    badgeBalance.className = "card-badge badge-green";
    valBalance.textContent = alloc.surplus.toFixed(1);
    valBalance.style.color = "#34d399";
    footerBalance.textContent = "Clean capacity buffer available";
  }

  // 2. Status Card
  const valStatus = document.getElementById("val-system-status");
  const systemDot = document.getElementById("system-dot");
  const badgeStatus = document.getElementById("badge-status");
  valStatus.textContent = alloc.status;
  
  if (alloc.status === "Normal") {
    valStatus.style.color = "#34d399";
    systemDot.style.background = "#10b981";
    systemDot.style.boxShadow = "0 0 10px #10b981";
    badgeStatus.textContent = "Optimal";
    badgeStatus.className = "card-badge badge-green";
  } else if (alloc.status === "Optimized") {
    valStatus.style.color = "#fbbf24";
    systemDot.style.background = "#f59e0b";
    systemDot.style.boxShadow = "0 0 10px #f59e0b";
    badgeStatus.textContent = "Managed";
    badgeStatus.className = "card-badge badge-savings";
  } else {
    valStatus.style.color = "#f87171";
    systemDot.style.background = "#ef4444";
    systemDot.style.boxShadow = "0 0 10px #ef4444";
    badgeStatus.textContent = "Critical";
    badgeStatus.className = "card-badge badge-red";
  }

  // 3. Cost & Savings Card
  if (alloc.cost_analysis) {
    document.getElementById("val-cost").textContent = alloc.cost_analysis.optimized_cost.toFixed(2);
    document.getElementById("cost-unoptimized").textContent = `$${alloc.cost_analysis.unoptimized_cost.toFixed(2)}/hr`;
    document.getElementById("cost-optimized").textContent = `$${alloc.cost_analysis.optimized_cost.toFixed(2)}/hr`;
    document.getElementById("cost-savings").textContent = `$${alloc.cost_analysis.estimated_saving.toFixed(2)}/hr`;
    document.getElementById("badge-savings").textContent = `Save $${alloc.cost_analysis.estimated_saving.toFixed(2)}`;
    
    const pct = alloc.cost_analysis.unoptimized_cost > 0
      ? Math.round((alloc.cost_analysis.estimated_saving / alloc.cost_analysis.unoptimized_cost) * 100)
      : 0;
    document.getElementById("cost-savings-sub").textContent = `~${pct}% energy cost reduction`;
  }

  // 4. Update Charts
  if (demandVsGenChart) {
    demandVsGenChart.data.datasets[0].data = [alloc.total_required, alloc.available_generation];
    demandVsGenChart.update();
  }

  if (allocationChart) {
    allocationChart.data.datasets[0].data = [alloc.critical, alloc.important, alloc.flexible];
    allocationChart.update();
  }

  // 5. Update Table Summary Badges
  const pctCrit = alloc.req_critical > 0 ? Math.round((alloc.critical / alloc.req_critical) * 100) : 100;
  const pctImp = alloc.req_important > 0 ? Math.round((alloc.important / alloc.req_important) * 100) : 100;
  const pctFlex = alloc.req_flexible > 0 ? Math.round((alloc.flexible / alloc.req_flexible) * 100) : 100;

  document.getElementById("badge-critical-total").textContent = `Critical: ${alloc.critical.toFixed(1)} kW (${pctCrit}%)`;
  document.getElementById("badge-important-total").textContent = `Important: ${alloc.important.toFixed(1)} kW (${pctImp}%)`;
  document.getElementById("badge-flexible-total").textContent = `Flexible: ${alloc.flexible.toFixed(1)} kW (${pctFlex}%)`;

  // 6. Render Dynamic Loads Table
  renderLoadsTable(alloc.loads);

  // 7. Update Alert Banner if not already explicitly showing surge/drop
  if (alloc.alert_message) {
    showAlert(alloc.alert_type, `System Alert [${alloc.status}]`, alloc.alert_message);
  }
}

function renderLoadsTable(loads) {
  const tbody = document.getElementById("loads-table-body");
  if (!tbody || !loads) return;

  tbody.innerHTML = "";

  loads.forEach(load => {
    const tr = document.createElement("tr");

    // Category Badge
    const catClass = load.category === "Critical" ? "tag-critical" :
                     load.category === "Important" ? "tag-important" : "tag-flexible";

    // Status Badge
    let statusClass = "status-active";
    if (load.status.includes("Managed")) statusClass = "status-managed";
    else if (load.status.includes("Shifted")) statusClass = "status-shifted";
    else if (load.status.includes("Reduced")) statusClass = "status-reduced";

    // Fulfillment Percentage
    const pct = load.required_kw > 0 ? Math.min(100, Math.round((load.allocated_kw / load.required_kw) * 100)) : 100;
    const barColor = pct >= 99 ? "#10b981" : pct >= 40 ? "#f59e0b" : "#a855f7";

    tr.innerHTML = `
      <td><strong>${escapeHtml(load.name)}</strong></td>
      <td><span class="table-badge ${catClass}">${load.category}</span></td>
      <td>Priority ${load.priority}</td>
      <td>${load.required_kw.toFixed(1)}</td>
      <td><strong>${load.allocated_kw.toFixed(1)}</strong></td>
      <td>
        <span class="progress-bar-bg">
          <span class="progress-bar-fill" style="width: ${pct}%; background: ${barColor};"></span>
        </span>
        ${pct}%
      </td>
      <td><span class="table-badge ${statusClass}">${escapeHtml(load.status)}</span></td>
    `;

    tbody.appendChild(tr);
  });
}

function showAlert(type, title, message) {
  const banner = document.getElementById("alert-banner");
  const alertIcon = document.getElementById("alert-icon");
  const alertTitle = document.getElementById("alert-title");
  const alertMsg = document.getElementById("alert-message");
  const alertTime = document.getElementById("alert-timestamp");

  banner.className = `alert alert-${type}`;
  alertIcon.textContent = type === "danger" ? "⚠" : type === "warning" ? "⚡" : "✓";
  alertTitle.textContent = title ? `${title}:` : "";
  alertMsg.textContent = message;
  
  const now = new Date();
  alertTime.textContent = now.toLocaleTimeString([], { hour: '2-digit', minute: '2-digit', second: '2-digit' });
}

function showBannerLoading(msg) {
  showAlert("warning", "Computing", msg);
}

function setConnectionActive(active) {
  const dot = document.getElementById("connection-dot");
  const txt = document.getElementById("connection-text");
  if (active) {
    dot.className = "status-dot pulse-green";
    txt.textContent = "Backend Active";
  } else {
    dot.className = "status-dot";
    dot.style.background = "#ef4444";
    txt.textContent = "Backend Disconnected";
  }
}

function getFormPayload() {
  return {
    current_demand: parseFloat(document.getElementById("input-current-demand").value),
    available_generation: parseFloat(document.getElementById("input-available-generation").value),
    temperature: parseFloat(document.getElementById("input-temperature").value),
    humidity: parseFloat(document.getElementById("input-humidity").value),
    occupancy: parseFloat(document.getElementById("input-occupancy").value),
    hour: parseInt(document.getElementById("input-hour").value),
    day_type: document.getElementById("input-day-type").value,
    user_activity: document.getElementById("input-user-activity").value
  };
}

function escapeHtml(str) {
  if (!str) return "";
  return str.replace(/[&<>'"]/g, 
    tag => ({ '&': '&amp;', '<': '&lt;', '>': '&gt;', "'": '&#39;', '"': '&quot;' }[tag] || tag)
  );
}
"""
print('JavaScript controller loaded. Characters:', len(JS_CONTENT))

---
## 9. Summary & Deployment

### How to Run this Project:
1. **Local Server**:
   ```bash
   python app.py
   ```
   Navigate to `http://127.0.0.1:5000`

2. **Live Public URL via Cloudflare Tunnel**:
   ```bash
   cloudflared tunnel --url http://127.0.0.1:5000
   ```

3. **GitHub Repository**:
   [https://github.com/257y1a6641-glitch/Electricity-Intelligence-AI.1](https://github.com/257y1a6641-glitch/Electricity-Intelligence-AI.1)